# Default Params

In [12]:
filename = "./melb_data.csv"
target_variable_name = "Price"

# Load Dataset

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = spark.read.csv(filename, header=True, inferSchema=True, sep=',')
data.show()

+----------+-------------------+-----+----+---------+------+-------+----------+--------+--------+--------+--------+---+--------+------------+---------+-----------+---------+----------+--------------------+-------------+
|    Suburb|            Address|Rooms|Type|    Price|Method|SellerG|      Date|Distance|Postcode|Bedroom2|Bathroom|Car|Landsize|BuildingArea|YearBuilt|CouncilArea|Lattitude|Longtitude|          Regionname|Propertycount|
+----------+-------------------+-----+----+---------+------+-------+----------+--------+--------+--------+--------+---+--------+------------+---------+-----------+---------+----------+--------------------+-------------+
|Abbotsford|       85 Turner St|    2|   h|1480000.0|     S| Biggin| 3/12/2016|     2.5|  3067.0|     2.0|     1.0|1.0|   202.0|        NULL|     NULL|      Yarra| -37.7996|  144.9984|Northern Metropol...|       4019.0|
|Abbotsford|    25 Bloomburg St|    2|   h|1035000.0|     S| Biggin| 4/02/2016|     2.5|  3067.0|     2.0|     1.0|0.0| 

In [7]:
data.count()

13580

In [8]:
data.describe().toPandas()

,summary,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,count,13580,13580,13580,13580,13580,13580,13580,13580,13580,...,13580,13518,13580,7130,8205,12211,13580,13580,13580,13580
1,mean,None,None,2.9379970544919,None,1075684.079455081,None,None,None,10.137776141384117,...,1.5342415316642122,1.6100754549489569,558.4161266568483,151.96764988779805,1964.6842169408897,None,-37.809202733431505,144.99521618777578,None,7454.417378497791
2,stddev,None,None,0.9557479384215565,None,639310.7242960163,None,None,None,5.868724943071715,...,0.6917117224588424,0.962633519245631,3990.669241109034,541.0145376263513,37.27376222396062,None,0.07925982260355832,0.10391556140730973,None,4378.581771795497
3,min,Abbotsford,1 Adelle Ct,1,h,85000.0,PI,@Realty,1/07/2017,0.0,...,0.0,0.0,0.0,0.0,1196.0,Banyule,-38.18255,144.43181,Eastern Metropolitan,249.0
4,max,Yarraville,9b Stewart St,10,u,9000000.0,VB,iTRAK,9/09/2017,48.1,...,8.0,10.0,433014.0,44515.0,2018.0,Yarra Ranges,-37.40853,145.52635,Western Victoria,21650.0


In [10]:
data.dtypes

[('Suburb', 'string'),
 ('Address', 'string'),
 ('Rooms', 'int'),
 ('Type', 'string'),
 ('Price', 'double'),
 ('Method', 'string'),
 ('SellerG', 'string'),
 ('Date', 'string'),
 ('Distance', 'double'),
 ('Postcode', 'double'),
 ('Bedroom2', 'double'),
 ('Bathroom', 'double'),
 ('Car', 'double'),
 ('Landsize', 'double'),
 ('BuildingArea', 'double'),
 ('YearBuilt', 'double'),
 ('CouncilArea', 'string'),
 ('Lattitude', 'double'),
 ('Longtitude', 'double'),
 ('Regionname', 'string'),
 ('Propertycount', 'double')]

In [11]:
data.printSchema()

root
 |-- Suburb: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Rooms: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Method: string (nullable = true)
 |-- SellerG: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Postcode: double (nullable = true)
 |-- Bedroom2: double (nullable = true)
 |-- Bathroom: double (nullable = true)
 |-- Car: double (nullable = true)
 |-- Landsize: double (nullable = true)
 |-- BuildingArea: double (nullable = true)
 |-- YearBuilt: double (nullable = true)
 |-- CouncilArea: string (nullable = true)
 |-- Lattitude: double (nullable = true)
 |-- Longtitude: double (nullable = true)
 |-- Regionname: string (nullable = true)
 |-- Propertycount: double (nullable = true)



In [14]:
from pyspark.sql.functions import * 
data.groupBy(target_variable_name).agg({'BuildingArea':'avg', 'Propertycount': 'avg'}).show()

+---------+------------------+------------------+
|    Price| avg(BuildingArea)|avg(Propertycount)|
+---------+------------------+------------------+
| 300000.0|             51.76| 7674.529411764706|
| 495000.0|137.53846153846155|           9056.55|
|1185000.0|             140.6| 8160.714285714285|
| 330000.0|41.980000000000004| 8083.538461538462|
| 532000.0|             103.5|            4357.0|
| 940500.0|              NULL|           11204.0|
| 452500.0|              54.0| 5934.666666666667|
| 546000.0| 95.66666666666667| 6626.571428571428|
| 474000.0|              62.5|            4939.0|
| 671000.0|107.33333333333333|            9819.0|
|1055000.0|135.57777777777778| 7882.058823529412|
| 677776.0|              64.0|           11204.0|
|2053000.0|              NULL|           11308.0|
| 975500.0|              NULL|            3464.0|
|1810000.0|160.33333333333334|           8531.75|
| 995000.0|             129.0|         5519.9375|
|1284000.0|              NULL|            6567.0|


# Check Cardinality

In [15]:
from pyspark.sql.functions import approxCountDistinct, countDistinct

"""
Note: approxCountDistinct and countDistinct can be used interchangeably. Only difference is the computation time. 

"approxCountDistinct" is useful for large datasets 
"countDistinct" for small and medium datasets.

"""

def cardinality_calculation(df, cut_off=1):
    cardinality = df.select(*[approxCountDistinct(c).alias(c) for c in df.columns])
    
    ## convert to pandas for efficient calculations
    final_cardinality_df = cardinality.toPandas().transpose()
    final_cardinality_df.reset_index(inplace=True) 
    final_cardinality_df.rename(columns={0:'Cardinality'}, inplace=True) 
    
    #select variables with cardinality of 1
    vars_selected = final_cardinality_df['index'][final_cardinality_df['Cardinality'] <= cut_off] 
    
    return final_cardinality_df, vars_selected

cardinality_df, cardinality_vars_selected = cardinality_calculation(data)

/usr/local/spark/python/pyspark/sql/functions.py:3796: FutureWarning: Deprecated in 2.1, use approx_count_distinct instead.
  warnings.warn("Deprecated in 2.1, use approx_count_distinct instead.", FutureWarning)


In [16]:
cardinality_df

,index,Cardinality
0,Suburb,318
1,Address,14421
2,Rooms,9
3,Type,3
4,Price,2328
5,Method,5
6,SellerG,277
7,Date,58
8,Distance,207
9,Postcode,207


# Missing Value Check

In [17]:
#missing values check
from pyspark.sql.functions import count, when, isnan, col

# miss_percentage is set to 80% as discussed in the book
def missing_calculation(df, miss_percentage=0.80):
    
    #checks for both NaN and null values
    missing = df.select(*[count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns])
    length_df = df.count()
    ## convert to pandas for efficient calculations
    final_missing_df = missing.toPandas().transpose()
    final_missing_df.reset_index(inplace=True) 
    final_missing_df.rename(columns={0:'missing_count'}, inplace=True) 
    final_missing_df['missing_percentage'] = final_missing_df['missing_count']/length_df
    
    #select variables with cardinality of 1
    vars_selected = final_missing_df['index'][final_missing_df['missing_percentage'] >= miss_percentage] 
    
    return final_missing_df, vars_selected

In [18]:
missing_calculation(data)

(            index  missing_count  missing_percentage
 0          Suburb              0            0.000000
 1         Address              0            0.000000
 2           Rooms              0            0.000000
 3            Type              0            0.000000
 4           Price              0            0.000000
 5          Method              0            0.000000
 6         SellerG              0            0.000000
 7            Date              0            0.000000
 8        Distance              0            0.000000
 9        Postcode              0            0.000000
 10       Bedroom2              0            0.000000
 11       Bathroom              0            0.000000
 12            Car             62            0.004566
 13       Landsize              0            0.000000
 14   BuildingArea           6450            0.474963
 15      YearBuilt           5375            0.395803
 16    CouncilArea           1369            0.100810
 17      Lattitude          

# Change all Missing Value to Mean

In [27]:
for column in data.columns:
    mean_value = data.select(mean(col(column)).alias('mean')).collect()[0]["mean"]
    mode_value = data.select(mode(col(column)).alias('mode')).collect()[0]['mode']
    if mean_value is not None:
        data = data.na.fill(mean_value, [column])
    else :
        data = data.na.fill(mode_value, [column])

In [28]:
missing_calculation(data)

(            index  missing_count  missing_percentage
 0          Suburb              0                 0.0
 1         Address              0                 0.0
 2           Rooms              0                 0.0
 3            Type              0                 0.0
 4           Price              0                 0.0
 5          Method              0                 0.0
 6         SellerG              0                 0.0
 7            Date              0                 0.0
 8        Distance              0                 0.0
 9        Postcode              0                 0.0
 10       Bedroom2              0                 0.0
 11       Bathroom              0                 0.0
 12            Car              0                 0.0
 13       Landsize              0                 0.0
 14   BuildingArea              0                 0.0
 15      YearBuilt              0                 0.0
 16    CouncilArea              0                 0.0
 17      Lattitude          

# Drop Useless Features

In [29]:
data = data.drop('Address')

In [30]:
data.show(5)

+----------+-----+----+---------+------+-------+---------+--------+--------+--------+--------+---+--------+------------------+------------------+-----------+---------+----------+--------------------+-------------+
|    Suburb|Rooms|Type|    Price|Method|SellerG|     Date|Distance|Postcode|Bedroom2|Bathroom|Car|Landsize|      BuildingArea|         YearBuilt|CouncilArea|Lattitude|Longtitude|          Regionname|Propertycount|
+----------+-----+----+---------+------+-------+---------+--------+--------+--------+--------+---+--------+------------------+------------------+-----------+---------+----------+--------------------+-------------+
|Abbotsford|    2|   h|1480000.0|     S| Biggin|3/12/2016|     2.5|  3067.0|     2.0|     1.0|1.0|   202.0|151.96764988779805|1964.6842169408897|      Yarra| -37.7996|  144.9984|Northern Metropol...|       4019.0|
|Abbotsford|    2|   h|1035000.0|     S| Biggin|4/02/2016|     2.5|  3067.0|     2.0|     1.0|0.0|   156.0|              79.0|            1900.0

# Label Encoding

In [31]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql.types import StringType

def index_string_columns(df):
    string_columns = [t[0] for t in df.dtypes if t[1] == 'string']
    for column in string_columns:
        indexer = StringIndexer(inputCol=column, outputCol=column+"_index")
        df = indexer.fit(df).transform(df)
    return df

In [37]:
data = index_string_columns(data)

IllegalArgumentException: requirement failed: Output column Suburb_index already exists.

In [34]:
data.printSchema()

root
 |-- Suburb: string (nullable = false)
 |-- Rooms: integer (nullable = true)
 |-- Type: string (nullable = false)
 |-- Price: double (nullable = false)
 |-- Method: string (nullable = false)
 |-- SellerG: string (nullable = false)
 |-- Date: string (nullable = false)
 |-- Distance: double (nullable = false)
 |-- Postcode: double (nullable = false)
 |-- Bedroom2: double (nullable = false)
 |-- Bathroom: double (nullable = false)
 |-- Car: double (nullable = false)
 |-- Landsize: double (nullable = false)
 |-- BuildingArea: double (nullable = false)
 |-- YearBuilt: double (nullable = false)
 |-- CouncilArea: string (nullable = false)
 |-- Lattitude: double (nullable = false)
 |-- Longtitude: double (nullable = false)
 |-- Regionname: string (nullable = false)
 |-- Propertycount: double (nullable = false)
 |-- Suburb_index: double (nullable = false)
 |-- Type_index: double (nullable = false)
 |-- Method_index: double (nullable = false)
 |-- SellerG_index: double (nullable = false)
 |

In [38]:
def variable_type(df):
    
    vars_list = df.dtypes
    char_vars = []
    num_vars = []
    for i in vars_list:
        if i[1] in ('string'):
            char_vars.append(i[0])
        else:
            num_vars.append(i[0])
    
    return char_vars, num_vars

In [40]:
char_vars, num_vars = variable_type(data)

data = data.select([c for c in data.columns if c not in char_vars])

In [42]:
data.dtypes

[('Rooms', 'int'),
 ('Price', 'double'),
 ('Distance', 'double'),
 ('Postcode', 'double'),
 ('Bedroom2', 'double'),
 ('Bathroom', 'double'),
 ('Car', 'double'),
 ('Landsize', 'double'),
 ('BuildingArea', 'double'),
 ('YearBuilt', 'double'),
 ('Lattitude', 'double'),
 ('Longtitude', 'double'),
 ('Propertycount', 'double'),
 ('Suburb_index', 'double'),
 ('Type_index', 'double'),
 ('Method_index', 'double'),
 ('SellerG_index', 'double'),
 ('Date_index', 'double'),
 ('CouncilArea_index', 'double'),
 ('Regionname_index', 'double')]

# Assemble Input Vectors

In [50]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

#assemble individual columns to one column - 'features'
def assemble_vectors(df, features_list, target_variable_name):
    stages = []
    #assemble vectors
    assembler = VectorAssembler(inputCols=features_list, outputCol='features')
    stages = [assembler]
    #select all the columns + target + newly created 'features' column
    selectedCols = [target_variable_name, 'features'] + features_list
    #use pipeline to process sequentially
    pipeline = Pipeline(stages=stages)
    #assembler model
    assembleModel = pipeline.fit(df)
    #apply assembler model on data
    df = assembleModel.transform(df).select(selectedCols)

    return df

# Split Data

In [54]:
train, test = data.randomSplit([0.7, 0.3], seed=12345)

In [55]:
#exclude target variable and select all other feature vectors
features_list = data.columns
#features_list = char_vars #this option is used only for ChiSqselector
features_list.remove(target_variable_name)

In [56]:
features_list

['Rooms',
 'Distance',
 'Postcode',
 'Bedroom2',
 'Bathroom',
 'Car',
 'Landsize',
 'BuildingArea',
 'YearBuilt',
 'Lattitude',
 'Longtitude',
 'Propertycount',
 'Suburb_index',
 'Type_index',
 'Method_index',
 'SellerG_index',
 'Date_index',
 'CouncilArea_index',
 'Regionname_index']

In [57]:
# apply the function on our dataframe
df_assemble = assemble_vectors(train, features_list, target_variable_name)

In [58]:
df_assemble.show(5)

+--------+--------------------+-----+--------+--------+--------+--------+---+--------+------------------+------------------+---------+------------------+-------------+------------+----------+------------+-------------+----------+-----------------+----------------+
|   Price|            features|Rooms|Distance|Postcode|Bedroom2|Bathroom|Car|Landsize|      BuildingArea|         YearBuilt|Lattitude|        Longtitude|Propertycount|Suburb_index|Type_index|Method_index|SellerG_index|Date_index|CouncilArea_index|Regionname_index|
+--------+--------------------+-----+--------+--------+--------+--------+---+--------+------------------+------------------+---------+------------------+-------------+------------+----------+------------+-------------+----------+-----------------+----------------+
| 85000.0|[1.0,6.4,3011.0,1...|    1|     6.4|  3011.0|     1.0|     1.0|0.0|     0.0|151.96764988779805|            2007.0| -37.7911|            144.89|       7570.0|        26.0|       1.0|         2.0| 

# Linear Regression

In [59]:
from pyspark.ml.regression import LinearRegression
reg = LinearRegression(featuresCol='features', labelCol='Price')
reg_model = reg.fit(df_assemble) # fit model

In [83]:
import pandas as pd

# Initialize a list to hold DataFrames
dfs = []

for k, v in df_assemble.schema["features"].metadata["ml_attr"]["attrs"].items():
    # Convert each dictionary in v to a DataFrame and add it to the list
    dfs.append(pd.DataFrame(v))

# Concatenate all the DataFrames in the list
features_df = pd.concat(dfs, ignore_index=True)

# print coefficient and intercept
print(reg_model.coefficients, reg_model.intercept)

features_df['coefficients'] = reg_model.coefficients

[211231.7627572584,-35870.006954687684,728.8952464397693,12676.240280852175,222479.51070893163,47340.18450781721,2.7734736958093853,50.52389205127527,-2445.084385773009,-1359227.4813994935,803047.1605612551,-12.50332185502953,-847.6159592606199,-201688.58515176407,-24221.058874300532,-27.592790175931423,-2385.5243326061327,-4560.652595081551,-44006.06051511781] -164559925.61804503


In [84]:
features_df

,idx,name,vals,coefficients
0,0,Rooms,NaN,2.112318e+05
1,1,Distance,NaN,-3.587001e+04
2,2,Postcode,NaN,7.288952e+02
3,3,Bedroom2,NaN,1.267624e+04
4,4,Bathroom,NaN,2.224795e+05
5,5,Car,NaN,4.734018e+04
6,6,Landsize,NaN,2.773474e+00
7,7,BuildingArea,NaN,5.052389e+01
8,8,YearBuilt,NaN,-2.445084e+03
9,9,Lattitude,NaN,-1.359227e+06


In [85]:
#prediction result
pred_result = reg_model.transform(df_assemble)

In [86]:
reg_summary = reg_model.summary

print('Mean Squared error', reg_summary.meanSquaredError)
print('Root mean squared error', reg_summary.rootMeanSquaredError)
print('Mean Absolute error', reg_summary.meanAbsoluteError)
print('Explained Variance', reg_summary.explainedVariance)
print('R squared', reg_summary.r2)

Mean Squared error 173182069402.07062
Root mean squared error 416151.4981374819
Mean Absolute error 280747.3984076761
Explained Variance 224837174484.1204
R squared 0.564890210624077


In [89]:
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(labelCol='Price', predictionCol='prediction', metricName='mse')

In [90]:
evaluator.evaluate(pred_result)

173182069402.07062

In [96]:
from pyspark.ml.evaluation import RegressionEvaluator

# Assuming 'lr_model' is your trained Linear Regression model
# and 'test_data' is your test data

# Generate predictions
assembled_test = assemble_vectors(test, features_list, target_variable_name)
predictions = reg_model.transform(assembled_test)

# Initialize evaluator
evaluator = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")

# Compute Root Mean Square Error (RMSE) on test data
rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)


Root Mean Squared Error (RMSE) on test data = 433611
